In [3]:
%matplotlib widget

In [14]:
from ctypes.wintypes import PUINT
from diagrams import Cluster, Diagram
from diagrams.gcp.analytics import BigQuery, Dataflow, PubSub
from diagrams.gcp.compute import AppEngine, Functions
from diagrams.gcp.database import BigTable
from diagrams.gcp.iot import IotCore
from diagrams.gcp.storage import GCS

with Diagram("RTIS Simulation", show=False, direction="TB"):
    pubsub = Functions("RTIS Data Initialization")

    with Cluster("RTIS Python Script"):
        [IotCore("Patient CT Slice"),
         IotCore("Spectrum Information"),
         IotCore("CBCT Geometry")] >> pubsub

    
    with Cluster("Simulation Files"):
            
        with Cluster("RTIS Files"):
            general_sim = [GCS("CT mhd"),
            GCS("CT range"),
            GCS("Spectrum dat"),GCS("CT density"),]
            with Cluster("Preset Files"):
                AppEngine("Detector OSF") 
                AppEngine("Bowtie Filter")
            pubsub >> general_sim
        
            gecco_sim = [
                        GCS("Attenuation Data")]
            
            pubsub >> gecco_sim

    with Cluster("RTIS Simulation Initialization"):

        # with Cluster("Primary Projections"):
        fc_sim = PubSub("Fastcat Simulation")
        # general_sim >>  fc_sim
        gecco_sim >> fc_sim

        # with Cluster("Secondary Projections"):
        #     gg_sim = PubSub("GGEMS Simulation")
        #     # general_sim >> gg_sim
        #     ggems_sim >> gg_sim

    with Cluster("Update Simulation"):
        update_data_sim = [AppEngine("Updated Spectrum"),
            AppEngine("Updated mAs"),
            AppEngine("Updated Scatter")]
        

    postprocess = Functions("RTIS Update and Reconstruction")
    fc_sim >> postprocess
    # gg_sim >> postprocess
    update_data_sim >> postprocess

    postprocess >> BigTable("RTIS CBCT")
            
            # with Cluster("Processing"):
            #     PubSub("Material File") # >> BigTable("bigtable")

            # with Cluster("Serverless"):
            #     Functions("func") >> AppEngine("appengine")

    # pubsub >> flow